# Low-Data Protein Language Model for Enzyme Function Prediction and Candidate Prioritization

**Cleaned research notebook — 12 October 2026**

This notebook consolidates the main experimental workflow and the latest saved evaluation artifacts. It removes repeated path searches, object-inspection cells, duplicated prediction diagnostics, notebook-source introspection, and temporary packaging cells from the original 188-cell notebook.

## Project scope

- Predict one of seven top-level Enzyme Commission (EC) classes from reviewed UniProt protein sequences.
- Compare conventional sequence-feature baselines with ESM-2 representations.
- Evaluate reduced-data learning curves, including cluster-aware sampling.
- Examine hierarchical EC 5.x subclass prediction.
- Audit a separate external candidate set for similarity to training proteins and assess generalization limitations.
- Build a candidate-prioritization workflow. Predictions are hypotheses, not experimental validation or proof of novel function.

## How to use this notebook

The main results and saved-artifact audit can be reviewed without running ESM-2. Embedding extraction and learning-curve cells are computationally expensive and are included as optional reproduction cells. Run them only when the required Kaggle dataset and split files are attached. Do not use the held-out test set to tune models.


## 1. Dataset and split design

The curated dataset contains **2,673 reviewed protein sequences** across top-level EC classes 1–7. MMseqs2 clustering used 30% minimum sequence identity and 80% coverage. The resulting 1,427 clusters were kept intact across train, validation, and test splits.

| Split | Proteins | Share |
|---|---:|---:|
| Training | 1,757 | 65.7% |
| Validation | 377 | 14.1% |
| Held-out test | 539 | 20.2% |
| **Total** | **2,673** | **100%** |

No cluster overlap was detected between the three partitions at the chosen clustering settings. This reduces close-homology leakage but does not prove that all remote homology or biological relatedness has been eliminated.


### 1.1 Load the split files (Kaggle)

Attach the `protein-set` dataset containing `train.csv`, `validation.csv`, and `test.csv`. This cell loads only train and validation initially; the test set is deliberately kept separate until final evaluation.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/s0ulraag/protein-set")
TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

for path in (TRAIN_PATH, VAL_PATH, TEST_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Required split not found: {path}. Attach the protein-set Kaggle dataset."
        )

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Training class counts:\n", train_df["label"].value_counts().sort_index())
print("Validation class counts:\n", val_df["label"].value_counts().sort_index())
assert len(train_df) == 1757 and len(val_df) == 377
assert train_df["Entry"].is_unique and val_df["Entry"].is_unique


## 2. ESM-2 embedding method

The representation model is `facebook/esm2_t12_35M_UR50D` (480-dimensional hidden representation). Long sequences are processed in windows of up to 1,000 residues with 200-residue overlap (stride 800). Special tokens are excluded. Overlapping residue vectors are averaged before pooling to one mean vector per protein.

**Cost warning:** the next two cells load the pretrained model and generate train/validation embeddings. This can take a substantial amount of GPU time. If the matching cached embedding files already exist, skip to Section 3. Do not rerun solely to reproduce already-saved results.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loading ESM-2 model...")
esm_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
esm_model.eval()
print("Device:", DEVICE)
print("Hidden size:", esm_model.config.hidden_size)


In [ ]:
import pandas as pd
import numpy as np

def sequence_length_report(df, name):
    lengths = df["Sequence"].astype(str).str.strip().str.len()

    print(f"\n{name} sequence-length report")
    print("-" * 40)
    print("Number of sequences:", len(lengths))
    print("Minimum length:", lengths.min())
    print("Median length:", lengths.median())
    print("Mean length:", round(lengths.mean(), 2))
    print("95th percentile:", lengths.quantile(0.95))
    print("99th percentile:", lengths.quantile(0.99))
    print("Maximum length:", lengths.max())
    print("Sequences longer than 1022:", (lengths > 1022).sum())
    print("Empty sequences:", (lengths == 0).sum())

    return lengths

train_lengths = sequence_length_report(train_df, "TRAIN")
val_lengths = sequence_length_report(val_df, "VALIDATION")

In [ ]:
from pathlib import Path
import numpy as np
import torch
import time
import json

# Settings
WINDOW_SIZE = 1000
OVERLAP = 200
STRIDE = WINDOW_SIZE - OVERLAP
BATCH_SIZE = 4  # conservative starting point for a 15.6 GB T4

OUTPUT_DIR = Path("/kaggle/working/esm2_embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def make_windows(sequence):
    sequence = str(sequence).strip().upper()
    if not sequence:
        raise ValueError("Encountered an empty protein sequence.")

    if len(sequence) <= WINDOW_SIZE:
        return [(0, sequence)]

    windows = []
    start = 0
    while start < len(sequence):
        chunk = sequence[start:start + WINDOW_SIZE]
        windows.append((start, chunk))
        if start + WINDOW_SIZE >= len(sequence):
            break
        start += STRIDE

    return windows


def embed_dataframe(df, split_name):
    sequences = df["Sequence"].astype(str).str.strip().str.upper().tolist()
    all_windows = []
    window_map = []

    # Record each window's protein and start position.
    for protein_idx, sequence in enumerate(sequences):
        windows = make_windows(sequence)
        for start, chunk in windows:
            window_map.append((protein_idx, start, len(chunk)))
            all_windows.append(chunk)

    # Store residue vectors per protein. Each residue is counted once.
    residue_sums = [None] * len(sequences)
    residue_counts = [None] * len(sequences)

    started = time.time()
    esm_model.eval()

    for batch_start in range(0, len(all_windows), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(all_windows))
        batch_sequences = all_windows[batch_start:batch_end]

        encoded = tokenizer(
            batch_sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=WINDOW_SIZE + 2
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.inference_mode():
            hidden = esm_model(**encoded).last_hidden_state

        attention = encoded["attention_mask"]

        for local_idx, global_idx in enumerate(range(batch_start, batch_end)):
            protein_idx, start, chunk_len = window_map[global_idx]

            # Extract amino-acid tokens only, excluding special tokens.
            token_vectors = hidden[local_idx, 1:chunk_len + 1].float().cpu().numpy()

            if residue_sums[protein_idx] is None:
                residue_sums[protein_idx] = np.zeros(
                    (len(sequences[protein_idx]), esm_model.config.hidden_size),
                    dtype=np.float64
                )
                residue_counts[protein_idx] = np.zeros(
                    len(sequences[protein_idx]), dtype=np.float32
                )

            end = start + chunk_len
            residue_sums[protein_idx][start:end] += token_vectors
            residue_counts[protein_idx][start:end] += 1

        if batch_end % 100 == 0 or batch_end == len(all_windows):
            print(f"{split_name}: processed {batch_end}/{len(all_windows)} windows")

    embeddings = []
    for i, sequence in enumerate(sequences):
        counts = residue_counts[i]
        if counts is None or np.any(counts == 0):
            raise RuntimeError(f"Some residues were not embedded in protein index {i}.")
        per_residue = residue_sums[i] / counts[:, None]
        embeddings.append(per_residue.mean(axis=0).astype(np.float32))

    embeddings = np.vstack(embeddings)

    output_file = OUTPUT_DIR / f"{split_name}_embeddings.npz"
    np.savez_compressed(
        output_file,
        embeddings=embeddings,
        labels=df["label"].to_numpy(),
        entries=df["Entry"].astype(str).to_numpy()
    )

    elapsed = time.time() - started
    print(f"\n{split_name} complete")
    print("Embedding matrix:", embeddings.shape)
    print("Elapsed seconds:", round(elapsed, 1))
    print("Saved to:", output_file)

    return embeddings


train_embeddings = embed_dataframe(train_df, "train")
val_embeddings = embed_dataframe(val_df, "validation")

print("\nAll embeddings generated.")
print("Train:", train_embeddings.shape)
print("Validation:", val_embeddings.shape)
print("Output directory:", OUTPUT_DIR)

## 3. Validation model selection

Validation macro F1 was used for model selection. The selected primary classifier is `StandardScaler + LinearSVC(C=2.0, class_weight="balanced")`. The saved project results report validation macro F1 ≈ 0.6463 for this configuration. The final test metrics below are historical saved results; do not repeatedly evaluate the test set while changing models.

The following cells compare logistic regression and linear SVM and tune the SVM regularization value using validation only. They require train and validation embeddings saved under `/kaggle/working/esm2_embeddings/`.


In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("/kaggle/working/esm2_embeddings")
RESULTS_DIR = Path("/kaggle/working/esm2_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load cached embeddings and labels
train_data = np.load(OUTPUT_DIR / "train_embeddings.npz")
val_data = np.load(OUTPUT_DIR / "validation_embeddings.npz")

X_train = train_data["embeddings"]
y_train = train_data["labels"].astype(int)

X_val = val_data["embeddings"]
y_val = val_data["labels"].astype(int)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        )
    ),
    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42,
            max_iter=10000
        )
    )
}

results = []
reports = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_val, predictions),
        "macro_f1": f1_score(y_val, predictions, average="macro"),
        "weighted_f1": f1_score(y_val, predictions, average="weighted")
    }
    results.append(metrics)

    reports[name] = classification_report(
        y_val, predictions, output_dict=True, zero_division=0
    )

    print("\nMetrics:")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float)
              else f"{key}: {value}")

    print("\nClassification report:")
    print(classification_report(y_val, predictions, zero_division=0))

    # Save confusion matrix for each model
    cm = confusion_matrix(y_val, predictions, labels=[1, 2, 3, 4, 5, 6, 7])
    pd.DataFrame(
        cm,
        index=[f"True_{i}" for i in range(1, 8)],
        columns=[f"Pred_{i}" for i in range(1, 8)]
    ).to_csv(RESULTS_DIR / f"{name.lower().replace(' ', '_')}_confusion_matrix.csv")

    # Save plot
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"{name} — ESM-2 Validation Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(7), range(1, 8))
    ax.set_yticks(range(7), range(1, 8))

    for i in range(7):
        for j in range(7):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / f"{name.lower().replace(' ', '_')}_confusion_matrix.png",
        dpi=160
    )
    plt.show()
    plt.close(fig)

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)
results_df.to_csv(RESULTS_DIR / "validation_model_comparison.csv", index=False)

with open(RESULTS_DIR / "classification_reports.json", "w") as f:
    json.dump(reports, f, indent=2)

print("\nMODEL COMPARISON — ranked by validation macro-F1")
print(results_df.to_string(index=False))
print("\nSaved results to:", RESULTS_DIR)
print("Held-out test set remains untouched.")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
import pandas as pd
import json
from pathlib import Path

EMBED_DIR = Path("/kaggle/working/esm2_embeddings")
RESULTS_DIR = Path("/kaggle/working/esm2_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_data = __import__("numpy").load(EMBED_DIR / "train_embeddings.npz")
val_data = __import__("numpy").load(EMBED_DIR / "validation_embeddings.npz")

X_train = train_data["embeddings"]
y_train = train_data["labels"].astype(int)
X_val = val_data["embeddings"]
y_val = val_data["labels"].astype(int)

results = []
models = {}

for c_value in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    model = make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=c_value,
            class_weight="balanced",
            random_state=42,
            max_iter=10000
        )
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    row = {
        "C": c_value,
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_val, predictions),
        "macro_f1": f1_score(y_val, predictions, average="macro"),
        "weighted_f1": f1_score(y_val, predictions, average="weighted")
    }

    results.append(row)
    models[c_value] = model

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)

results_df.to_csv(
    RESULTS_DIR / "esm2_svm_c_tuning.csv",
    index=False
)

print(results_df.to_string(index=False))

best_c = float(results_df.iloc[0]["C"])
print(f"\nBest C by validation macro-F1: {best_c}")
print("Best validation macro-F1:",
      round(float(results_df.iloc[0]["macro_f1"]), 4))

with open(RESULTS_DIR / "esm2_svm_tuning_summary.json", "w") as f:
    json.dump({
        "selection_metric": "validation_macro_f1",
        "best_C": best_c,
        "best_validation_macro_f1":
            float(results_df.iloc[0]["macro_f1"]),
        "test_set_used": False,
        "all_results": results
    }, f, indent=2)

print("Tuning results saved to:", RESULTS_DIR)
print("Held-out test set remains untouched.")

In [ ]:
from pathlib import Path
import json

results_dir = Path("/kaggle/working/esm2_results")
results_dir.mkdir(parents=True, exist_ok=True)

selected_model = {
    "experiment": "ESM-2 embeddings + Linear SVM",
    "embedding_model": "facebook/esm2_t12_35M_UR50D",
    "embedding_dimension": 480,
    "classifier": "LinearSVC",
    "selected_C": 2.0,
    "class_weight": "balanced",
    "selection_metric": "validation_macro_f1",
    "validation_accuracy": 0.660477,
    "validation_balanced_accuracy": 0.652247,
    "validation_macro_f1": 0.646287,
    "validation_weighted_f1": 0.650582,
    "training_records": 1757,
    "validation_records": 377,
    "test_set_used_for_selection": False,
    "random_state": 42
}

output_path = results_dir / "selected_esm2_svm_config.json"

with open(output_path, "w") as f:
    json.dump(selected_model, f, indent=2)

print("Selected model configuration saved:")
print(output_path)
print(json.dumps(selected_model, indent=2))

## 4. Held-out test results (saved final evaluation)

The final selected model was fitted on train plus validation (2,134 proteins) and evaluated once on the 539-protein held-out test split. The saved results are:

| Metric | Test result |
|---|---:|
| Accuracy | 71.43% |
| Balanced accuracy | 71.12% |
| Macro F1 | 0.7035 |
| Weighted F1 | 0.7093 |

Per-class F1: EC 1 = 0.60, EC 2 = 0.82, EC 3 = 0.72, EC 4 = 0.58, EC 5 = 0.49, EC 6 = 0.81, EC 7 = 0.90. EC class 5 is a weaker class and is a motivation for the hierarchical experiment. These are broad top-level EC predictions, not proof of a complete EC number or experimentally verified function.

Tracked output files include `protein_project/results/esm2_final/final_test_metrics.json`, `final_test_classification_report.csv`, `final_test_confusion_matrix.csv`, and `final_test_confusion_matrix.png`.


## 5. Low-data experiments

These experiments use existing embeddings and a fixed validation set; they do **not** evaluate the test set. Three seeds (42, 123, 2026) were used at 10%, 25%, and 50%; the 100% row is the full training split and therefore does not vary across seeds.

### 5.1 Protein-level stratified sampling

| Training fraction | Mean proteins | Linear SVM macro F1 | Logistic regression macro F1 |
|---:|---:|---:|---:|
| 10% | 173 | 0.5281 | 0.5336 |
| 25% | 436 | 0.5642 | 0.5912 |
| 50% | 877 | 0.5681 | 0.5878 |
| 100% | 1,757 | 0.6463 | 0.6333 |

### 5.2 Cluster-aware sampling

Whole training clusters were sampled to avoid splitting related proteins within the training subset. Protein counts vary because cluster sizes differ.

| Training fraction | Mean proteins | Mean clusters | Linear SVM macro F1 | Logistic regression macro F1 |
|---:|---:|---:|---:|---:|
| 10% | 187.3 | 103.0 | 0.4675 | 0.4336 |
| 25% | 440.0 | 234.3 | 0.5753 | 0.5743 |
| 50% | 879.7 | 466.0 | 0.5825 | 0.5982 |
| 100% | 1,757 | 914 | 0.6463 | 0.6333 |

The trend improves overall with more data, but is not strictly monotonic at each intermediate fraction. These are validation results on the same fixed split, not independent evidence of generalization to entirely unseen enzyme families.

The next two cells reproduce these learning curves using existing `.npz` embeddings. They can be skipped if the summaries are already sufficient. They expect `/kaggle/working/esm2_embeddings/{train,validation}_embeddings.npz` and, for cluster-aware sampling, `cluster_id` in the attached split CSVs.


In [ ]:
# LOW-DATA LEARNING-CURVE EXPERIMENT
# Uses existing ESM-2 embeddings; does not retrain ESM-2.
# The test set is not loaded or evaluated in this experiment.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ---------- Paths ----------
EMB_DIR = Path("/kaggle/working/esm2_embeddings")
OUT_DIR = Path("/kaggle/working/esm2_results/low_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Load training and validation embeddings ----------
def load_embeddings(path):
    # These are the embedding files generated by your own notebook.
    with np.load(path, allow_pickle=True) as data:
        X = np.asarray(data["embeddings"], dtype=np.float32)
        y = np.asarray(data["labels"], dtype=np.int64)
    return X, y

X_train, y_train = load_embeddings(
    EMB_DIR / "train_embeddings.npz"
)
X_val, y_val = load_embeddings(
    EMB_DIR / "validation_embeddings.npz"
)

assert X_train.ndim == 2 and X_val.ndim == 2
assert X_train.shape[1] == X_val.shape[1] == 480
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert set(np.unique(y_train)) == set(np.unique(y_val))

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Training class counts:", dict(zip(
    *np.unique(y_train, return_counts=True)
)))
print("Validation class counts:", dict(zip(
    *np.unique(y_val, return_counts=True)
)))

# ---------- Experiment settings ----------
fractions = [0.10, 0.25, 0.50, 1.00]
seeds = [42, 123, 2026]

models = {
    "ESM-2 + Linear SVM": lambda: make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=2.0,
            class_weight="balanced",
            random_state=42,
            max_iter=10000
        )
    ),
    "ESM-2 + Logistic Regression": lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
}

classes = np.sort(np.unique(y_train))
records = []

# Stratified sampling: preserve representation of every EC class.
# Existing cluster-aware train/validation separation remains intact.
for fraction in fractions:
    for seed in seeds:
        rng = np.random.default_rng(seed)
        selected = []

        for label in classes:
            class_indices = np.flatnonzero(y_train == label)
            n_take = max(1, int(np.floor(
                len(class_indices) * fraction
            )))
            n_take = min(n_take, len(class_indices))

            chosen = rng.choice(
                class_indices, size=n_take, replace=False
            )
            selected.extend(chosen.tolist())

        selected = np.array(selected, dtype=int)
        rng.shuffle(selected)

        X_sub = X_train[selected]
        y_sub = y_train[selected]

        for model_name, make_model in models.items():
            model = make_model()
            model.fit(X_sub, y_sub)
            pred = model.predict(X_val)

            records.append({
                "fraction": fraction,
                "training_proteins": len(selected),
                "seed": seed,
                "model": model_name,
                "accuracy": accuracy_score(y_val, pred),
                "balanced_accuracy": balanced_accuracy_score(
                    y_val, pred
                ),
                "macro_f1": f1_score(
                    y_val, pred, average="macro"
                ),
                "weighted_f1": f1_score(
                    y_val, pred, average="weighted"
                )
            })

        print(
            f"Completed fraction={fraction:.0%}, "
            f"seed={seed}, proteins={len(selected)}"
        )

# ---------- Save individual runs ----------
results = pd.DataFrame(records)
csv_path = OUT_DIR / "low_data_runs.csv"
results.to_csv(csv_path, index=False)

# ---------- Aggregate across random seeds ----------
summary = (
    results.groupby(["fraction", "training_proteins", "model"])
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        accuracy_mean=("accuracy", "mean"),
        weighted_f1_mean=("weighted_f1", "mean")
    )
    .reset_index()
)

summary_path = OUT_DIR / "low_data_summary.csv"
summary.to_csv(summary_path, index=False)

# ---------- Plot learning curves ----------
fig, ax = plt.subplots(figsize=(9, 6))

for model_name in models:
    subset = summary[summary["model"] == model_name].sort_values(
        "fraction"
    )
    x = subset["fraction"].to_numpy() * 100
    mean = subset["macro_f1_mean"].to_numpy()
    std = subset["macro_f1_std"].fillna(0).to_numpy()

    ax.plot(x, mean, marker="o", label=model_name)
    ax.fill_between(
        x,
        np.maximum(0, mean - std),
        np.minimum(1, mean + std),
        alpha=0.15
    )

ax.set_xlabel("Training data used (%)")
ax.set_ylabel("Validation macro F1")
ax.set_title("ESM-2 Low-Data Learning Curve")
ax.set_xticks([10, 25, 50, 100])
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plot_path = OUT_DIR / "low_data_learning_curve.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("\n--- SUMMARY: MEAN ACROSS THREE SEEDS ---")
print(summary.round(4).to_string(index=False))

print("\nSaved files:")
print(csv_path)
print(summary_path)
print(plot_path)

In [ ]:
# CLUSTER-AWARE LOW-DATA EXPERIMENT
# Uses existing ESM-2 embeddings and cluster IDs.
# Does not load or evaluate the test set.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

DATA_DIR = Path("/kaggle/input/datasets/s0ulraag/protein-set")
EMB_DIR = Path("/kaggle/working/esm2_embeddings")
OUT_DIR = Path("/kaggle/working/esm2_results/cluster_low_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load split CSVs and their existing cluster assignments.
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")

assert train_df["cluster_id"].notna().all()
assert val_df["cluster_id"].notna().all()

# Labels should be the existing numeric top-level EC class labels.
LABEL_COL = "label"
assert LABEL_COL in train_df.columns, (
    f"Expected '{LABEL_COL}'. Available columns: {train_df.columns.tolist()}"
)

# Load embeddings and saved entry IDs.
def load_archive(name):
    with np.load(
        EMB_DIR / f"{name}_embeddings.npz",
        allow_pickle=True
    ) as data:
        return (
            np.asarray(data["embeddings"], dtype=np.float32),
            np.asarray(data["labels"], dtype=np.int64),
            np.asarray(data["entries"]).astype(str)
        )

X_train_raw, y_train_raw, train_entries = load_archive("train")
X_val_raw, y_val_raw, val_entries = load_archive("validation")

# Align embedding rows to CSV order using UniProt Entry IDs.
def align_to_csv(df, X, y, entries, split_name):
    if df["Entry"].astype(str).duplicated().any():
        raise ValueError(f"Duplicate Entry IDs in {split_name} CSV.")
    if len(set(entries)) != len(entries):
        raise ValueError(f"Duplicate embedding Entry IDs in {split_name}.")

    index_by_entry = {entry: i for i, entry in enumerate(entries)}
    csv_entries = df["Entry"].astype(str).tolist()

    missing = set(csv_entries) - set(index_by_entry)
    extra = set(index_by_entry) - set(csv_entries)
    if missing or extra:
        raise ValueError(
            f"{split_name} ID mismatch: missing={len(missing)}, "
            f"extra={len(extra)}"
        )

    order = np.array([index_by_entry[e] for e in csv_entries])
    X_aligned = X[order]
    y_aligned = y[order]
    y_csv = df[LABEL_COL].to_numpy(dtype=int)

    if not np.array_equal(y_aligned, y_csv):
        raise ValueError(
            f"{split_name} labels do not match embedding labels. "
            "Stop and inspect the label mapping."
        )

    return X_aligned, y_csv

X_train, y_train = align_to_csv(
    train_df, X_train_raw, y_train_raw, train_entries, "train"
)
X_val, y_val = align_to_csv(
    val_df, X_val_raw, y_val_raw, val_entries, "validation"
)

classes = np.sort(np.unique(y_train))
assert np.array_equal(classes, np.sort(np.unique(y_val)))

print("Aligned training:", X_train.shape)
print("Aligned validation:", X_val.shape)
print("EC classes:", classes.tolist())
print("Training proteins:", len(train_df))
print("Training clusters:", train_df["cluster_id"].nunique())

# Each cluster is sampled as an indivisible unit.
cluster_groups = {
    str(cluster): np.asarray(indices, dtype=int)
    for cluster, indices in train_df.groupby("cluster_id").groups.items()
}

cluster_labels = {}
for cluster, idx in cluster_groups.items():
    cluster_labels[cluster] = set(y_train[idx].tolist())

clusters_by_class = {
    label: [
        cluster for cluster, labels in cluster_labels.items()
        if label in labels
    ]
    for label in classes
}

for label, candidates in clusters_by_class.items():
    if not candidates:
        raise ValueError(f"No training cluster contains class {label}")

def sample_whole_clusters(fraction, seed):
    """Select whole clusters, target approximate protein fraction,
    and ensure all EC classes are represented where possible."""
    if fraction == 1.0:
        return np.arange(len(train_df)), list(cluster_groups.keys())

    rng = np.random.default_rng(seed)
    target_n = int(np.ceil(len(train_df) * fraction))
    selected_clusters = []

    # First choose one cluster containing each class.
    # A multi-label cluster may satisfy more than one class.
    class_order = list(classes)
    rng.shuffle(class_order)

    for label in class_order:
        if any(label in cluster_labels[c] for c in selected_clusters):
            continue

        candidates = [
            c for c in clusters_by_class[label]
            if c not in selected_clusters
        ]
        if candidates:
            selected_clusters.append(
                candidates[int(rng.integers(len(candidates)))]
            )

    # Add randomly ordered whole clusters until the target is reached.
    remaining = [
        c for c in cluster_groups if c not in selected_clusters
    ]
    rng.shuffle(remaining)

    selected_indices = np.concatenate([
        cluster_groups[c] for c in selected_clusters
    ]) if selected_clusters else np.array([], dtype=int)

    for cluster in remaining:
        if len(selected_indices) >= target_n:
            break
        selected_clusters.append(cluster)
        selected_indices = np.concatenate([
            selected_indices, cluster_groups[cluster]
        ])

    selected_indices = np.sort(selected_indices)

    represented = set(y_train[selected_indices].tolist())
    if represented != set(classes):
        raise ValueError(
            f"Not all classes represented at {fraction}: {represented}"
        )

    return selected_indices, selected_clusters

models = {
    "ESM-2 + Linear SVM": lambda: make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=2.0,
            class_weight="balanced",
            random_state=42,
            max_iter=10000
        )
    ),
    "ESM-2 + Logistic Regression": lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
}

fractions = [0.10, 0.25, 0.50, 1.00]
seeds = [42, 123, 2026]
records = []

for fraction in fractions:
    for seed in seeds:
        idx, selected_clusters = sample_whole_clusters(fraction, seed)
        X_sub, y_sub = X_train[idx], y_train[idx]

        for model_name, make_model in models.items():
            model = make_model()
            model.fit(X_sub, y_sub)
            pred = model.predict(X_val)

            records.append({
                "fraction_target": fraction,
                "seed": seed,
                "training_proteins": len(idx),
                "training_clusters": len(selected_clusters),
                "model": model_name,
                "accuracy": accuracy_score(y_val, pred),
                "balanced_accuracy": balanced_accuracy_score(y_val, pred),
                "macro_f1": f1_score(y_val, pred, average="macro"),
                "weighted_f1": f1_score(
                    y_val, pred, average="weighted"
                )
            })

        print(
            f"Target={fraction:.0%}; seed={seed}; "
            f"proteins={len(idx)}/{len(train_df)}; "
            f"clusters={len(selected_clusters)}"
        )

results = pd.DataFrame(records)
results.to_csv(OUT_DIR / "cluster_low_data_runs.csv", index=False)

summary = (
    results.groupby(["fraction_target", "model"])
    .agg(
        proteins_mean=("training_proteins", "mean"),
        clusters_mean=("training_clusters", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        accuracy_mean=("accuracy", "mean"),
        weighted_f1_mean=("weighted_f1", "mean")
    )
    .reset_index()
)
summary.to_csv(OUT_DIR / "cluster_low_data_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
for model_name in models:
    sub = summary[summary["model"] == model_name].sort_values(
        "fraction_target"
    )
    x = sub["fraction_target"].to_numpy() * 100
    mean = sub["macro_f1_mean"].to_numpy()
    std = sub["macro_f1_std"].fillna(0).to_numpy()

    ax.plot(x, mean, marker="o", label=model_name)
    ax.fill_between(
        x, np.maximum(0, mean - std),
        np.minimum(1, mean + std), alpha=0.15
    )

ax.set_xlabel("Target training fraction (%)")
ax.set_ylabel("Validation macro F1")
ax.set_title("Cluster-Aware Low-Data Learning Curve")
ax.set_xticks([10, 25, 50, 100])
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(
    OUT_DIR / "cluster_low_data_learning_curve.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

print("\n--- CLUSTER-AWARE SUMMARY ---")
print(summary.round(4).to_string(index=False))
print("\nResults saved to:", OUT_DIR)

## 6. Hierarchical EC class 5 experiment

This is a **conditional** subclass experiment: Stage 2 receives proteins already known to be top-level EC class 5. It does not mean the complete two-stage pipeline achieves the same score.

| Stage 2 metric | Validation result |
|---|---:|
| Accuracy | 88.46% (46/52) |
| Balanced accuracy | 88.91% |
| Macro F1 | 0.8566 |
| Weighted F1 | 0.8772 |

End-to-end routing was the bottleneck: Stage 1 routed only 25 of the 52 true class-5 validation proteins to Stage 2. End-to-end subclass accuracy on those 52 proteins was therefore 25/52 = 48.08%. EC 5.5 had only two training examples and no validation examples, so it could not be meaningfully evaluated. The conditional Stage 2 result must not be presented as end-to-end performance.

Saved artifacts tracked in the repository include `validation_complete_hierarchical_metrics.json`, `validation_complete_hierarchical_pipeline.csv`, `validation_ec5_hierarchical_comparison.csv`, `validation_ec5_hierarchical_predictions.csv`, and `validation_ec5_subclass_analysis.csv` under `protein_project/results/`.


## 7. External evaluation and sequence-similarity audit

The current external collection has 222 single-top-level-class candidates before the similarity screen. The local alignment audit retained 138 lower-local-similarity proteins and excluded 84 potentially related/very-close proteins. The filter is not class-neutral: it disproportionately excludes class 7, so results on the retained set are subject to selection bias.

### 7.1 Original-method external metrics

The classifier was `StandardScaler + RBF SVC(C=10, class_weight="balanced")`, with the original 480-dimensional embeddings and overlap-corrected pooling. The 138 retained proteins produced:

| Metric | Result |
|---|---:|
| Accuracy | 46.38% |
| Balanced accuracy | 55.99% |
| Macro F1 across all 7 classes | 0.3538 |
| Weighted F1 | 0.5076 |

Only classes 1, 2, 3, 6, and 7 are represented in this retained set; classes 4 and 5 have zero support. Class 2 recall was 0, and class 7 recall was 0.375. This is a preliminary external diagnostic, not a balanced independent benchmark. It should not be compared as if it were the same protocol as the held-out internal test result: the external classifier and sampling process differ.

### 7.2 Similarity-filtered excluded subset

The excluded 84 proteins contain 11 class-6 and 73 class-7 records. The classifier correctly predicted all 11 class-6 proteins but only 2 of 73 class-7 proteins. This subset is explicitly similarity-filtered and highly family-redundant; it is diagnostic evidence of limitations, not an unbiased benchmark.


### 7.3 Load and inspect the downloaded results bundle

To reproduce the artifact audit in a fresh Kaggle session, upload `enzyme_project_results_bundle.zip` to the notebook session or attach it as a Kaggle input dataset. This cell searches for the ZIP and extracts it to `/kaggle/working/enzyme_project_results_bundle/` if necessary. If the files already exist under `/kaggle/working/enzyme_candidates/`, those are used directly.

In [ ]:
from pathlib import Path
import zipfile
import json
import pandas as pd

WORK = Path("/kaggle/working")
BUNDLE_DIR = WORK / "enzyme_project_results_bundle"
ZIP_CANDIDATES = list(WORK.glob("enzyme_project_results_bundle.zip"))

if not (BUNDLE_DIR / "enzyme_candidates").exists() and ZIP_CANDIDATES:
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_CANDIDATES[0]) as zf:
        zf.extractall(BUNDLE_DIR)

# Prefer the current working output layout, then fall back to the extracted bundle.
CURRENT = WORK / "enzyme_candidates"
BUNDLED = BUNDLE_DIR / "enzyme_candidates"
ROOT = CURRENT if (CURRENT / "similarity_check").exists() else BUNDLED
SIM_DIR = ROOT / "similarity_check"
print("Using artifact directory:", SIM_DIR)
if not SIM_DIR.exists():
    print("External artifacts not found. Upload/extract the results bundle before running this section.")
else:
    metrics_path = SIM_DIR / "original_method_evaluation" / "external_metrics_original_embeddings.json"
    if not metrics_path.exists():
        metrics_path = SIM_DIR / "external_eval_metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2))

    predictions_path = SIM_DIR / "original_method_evaluation" / "external_predictions_original_embeddings.csv"
    if not predictions_path.exists():
        predictions_path = SIM_DIR / "external_eval_predictions.csv"
    predictions = pd.read_csv(predictions_path)
    print("\nPrediction table shape:", predictions.shape)
    print("Columns:", predictions.columns.tolist())
    if {"actual_class", "predicted_class"}.issubset(predictions.columns):
        print("\nTrue class counts:\n", predictions["actual_class"].value_counts().sort_index())
        print("\nPrediction counts:\n", predictions["predicted_class"].value_counts().sort_index())
        if "correct" in predictions.columns:
            print("Correct:", int(predictions["correct"].astype(bool).sum()), "/", len(predictions))
        elif "prediction_correct" in predictions.columns:
            print("Correct:", int(predictions["prediction_correct"].astype(bool).sum()), "/", len(predictions))


## 8. Candidate-pool inventory and safeguards

The downloaded bundle includes raw and filtered microbial candidate pools. The filtered microbial pool has 500 rows; the EC-balanced filtered collection has 482 rows. These are **candidate pools**, not confirmed novel enzymes. Before ranking candidates, inspect accession overlap, exact-sequence overlap, annotation quality, family redundancy, and similarity to the training set. Local pairwise alignment was used as a screening diagnostic, not as a substitute for a full-scale homology search.

Do not claim novelty from a low local alignment score alone. Novel-function claims require careful sequence/structure/context analysis and ideally experimental validation.


In [ ]:
from pathlib import Path
import pandas as pd

candidate_files = {
    "microbial_candidate_pool_raw.csv": ROOT / "microbial_candidate_pool_raw.csv",
    "microbial_candidate_pool_filtered.csv": ROOT / "microbial_candidate_pool_filtered.csv",
    "microbial_ec_balanced_raw.csv": ROOT / "microbial_ec_balanced_raw.csv",
    "microbial_ec_balanced_filtered.csv": ROOT / "microbial_ec_balanced_filtered.csv",
    "microbial_external_eval_single_class.csv": ROOT / "microbial_external_eval_single_class.csv",
}

summary_rows = []
for name, path in candidate_files.items():
    if path.exists():
        df = pd.read_csv(path)
        summary_rows.append({
            "file": name,
            "rows": len(df),
            "columns": len(df.columns),
            "unique_accessions": df["Entry"].nunique() if "Entry" in df else None,
            "has_sequence": "Sequence" in df,
        })
    else:
        print("Not found:", path)

if summary_rows:
    display(pd.DataFrame(summary_rows))


## 9. Limitations and current status

### Completed

- Reviewed-UniProt dataset preparation and split integrity checks.
- MMseqs2 cluster-aware train/validation/test partitions.
- Amino-acid composition and k-mer baselines.
- ESM-2 embeddings and validation-based classifier selection.
- Held-out internal test evaluation.
- Protein-level and cluster-aware low-data learning curves.
- Conditional EC class-5 subclass analysis and end-to-end routing audit.
- External candidate collection, similarity filtering, and preliminary generalization diagnostics.

### Still to complete

1. Verify that the cleaned notebook runs from beginning to end using the required Kaggle inputs and cached artifacts; do not rerun expensive embedding extraction solely to reproduce saved results.
2. Recover and version the low-data CSV summaries and figures if they can be retrieved and their provenance verified.
3. Build a transparent candidate-ranking table using predicted class, model confidence where available, sequence similarity, and annotation evidence.
4. Document the candidate-ranking criteria and limitations. Do not describe candidates as novel without adequate supporting evidence.
5. Review the final README, progress log, notebook, and result artifacts together, then inspect the exact Git diff before committing.

### Evaluation safeguards

- Do not tune on the held-out test set.
- Keep conditional class-5 metrics separate from end-to-end results.
- State that the external retained set is imbalanced and affected by similarity filtering.
- Treat model outputs as computational predictions, not experimental confirmation.
